# Permuted-Label Control for Deployed BBB Models

This notebook runs a direct permuted-label sanity check for the three models connected to the Streamlit app:

- PaDEL + Morgan / LightGBM / duplicate-aware seed 5
- PaDEL + Morgan / Extra Trees / duplicate-aware seed 5
- PaDEL + Morgan + ChemBERTa embeddings / XGBoost / scaffold-CV fold 1

The saved deployed pipelines are loaded only to reuse their tuned hyperparameters and preprocessing structure. For each permutation seed, a cloned copy of the pipeline is retrained from scratch on shuffled training labels and evaluated on shuffled held-out labels from the same saved validation split. If the pipeline were learning from leakage artifacts, performance could remain above chance even after labels are randomized. Performance near 0.5 supports that the deployed model workflows do not recover signal from randomized labels.

In [ ]:
# Core imports used throughout the notebook.
from pathlib import Path
import sys

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.metrics import balanced_accuracy_score, roc_auc_score, average_precision_score


# Locate the repository root whether this notebook is run from the repo root
# or from inside brainroute_ml_validation/.
def find_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, cwd.parent, cwd / "brainroute_ml_validation"]:
        if (candidate / "brainroute_ml_validation" / "configs" / "validation_config.yaml").exists():
            return candidate
        if candidate.name == "brainroute_ml_validation" and (candidate / "configs" / "validation_config.yaml").exists():
            return candidate.parent
    raise FileNotFoundError("Could not locate repository root.")


REPO_ROOT = find_repo_root()
VALIDATION_ROOT = REPO_ROOT / "brainroute_ml_validation"
sys.path.insert(0, str(REPO_ROOT))

from brainroute_ml_validation.src.features import load_feature_view
from brainroute_ml_validation.src.modeling import split_data_for_feature_view
from brainroute_ml_validation.src.utils import load_config

cfg = load_config(VALIDATION_ROOT / "configs" / "validation_config.yaml")
REPORT_DIR = VALIDATION_ROOT / "reports"
FIGURE_DIR = REPORT_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repository root: {REPO_ROOT}")

In [ ]:
# Models selected because these are the current Streamlit deployment models.
# The split values match the validation split/fold used by each saved model file.
DEPLOYED_MODELS = [
    {
        "label": "PaDEL + Morgan / LightGBM",
        "feature_view": "padel_morgan",
        "model": "lightgbm",
        "split": "duplicate_aware_seed5",
        "model_path": VALIDATION_ROOT / "models" / "padel_morgan__lightgbm__duplicate_aware_seed5.joblib",
    },
    {
        "label": "PaDEL + Morgan / Extra Trees",
        "feature_view": "padel_morgan",
        "model": "extra_trees",
        "split": "duplicate_aware_seed5",
        "model_path": VALIDATION_ROOT / "models" / "padel_morgan__extra_trees__duplicate_aware_seed5.joblib",
    },
    {
        "label": "PaDEL + Morgan + ChemBERTa / XGBoost",
        "feature_view": "padel_morgan_embeddings",
        "model": "xgboost",
        "split": "scaffold_cv_fold1",
        "model_path": VALIDATION_ROOT / "models" / "padel_morgan_embeddings__xgboost__scaffold_cv_fold1.joblib",
    },
]

# Fixed seeds make the random-label controls exactly reproducible.
PERMUTATION_SEEDS = [11, 22, 33]


# Set to an integer such as 1000 if you only want a quick local smoke test.
MAX_TRAIN_ROWS = None
MAX_TEST_ROWS = None

## Helper Functions

In [ ]:
def maybe_subsample(X_train, X_test, y_train, y_test, max_train_rows=None, max_test_rows=None, seed=42):
    """Optionally subsample train/test rows for a fast smoke test while preserving labels."""
    rng = np.random.default_rng(seed)
    if max_train_rows is not None and len(X_train) > max_train_rows:
        train_idx = np.sort(rng.choice(len(X_train), size=max_train_rows, replace=False))
        X_train = X_train.iloc[train_idx]
        y_train = y_train[train_idx]
    if max_test_rows is not None and len(X_test) > max_test_rows:
        test_idx = np.sort(rng.choice(len(X_test), size=max_test_rows, replace=False))
        X_test = X_test.iloc[test_idx]
        y_test = y_test[test_idx]
    return X_train, X_test, y_train, y_test


def predict_positive_probability(model, X):
    """Return BBB+ probability if available; otherwise return a decision score."""
    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X)
        return proba[:, 1] if proba.shape[1] > 1 else proba[:, 0]
    if hasattr(model, "decision_function"):
        return model.decision_function(X)
    return model.predict(X)


def permute_train_and_test_labels(y_train, y_test, seed):
    """Shuffle labels after the split so no feature-label relationship remains."""
    rng = np.random.default_rng(seed) #shuffles
    return rng.permutation(y_train), rng.permutation(y_test)


def evaluate_predictions(y_true, y_pred, y_score):
    """Compute chance-sensitive metrics for the permuted-label control."""
    metrics = {
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "roc_auc": np.nan,
        "auprc": np.nan,
    }
    if len(np.unique(y_true)) > 1:
        metrics["roc_auc"] = roc_auc_score(y_true, y_score)
        metrics["auprc"] = average_precision_score(y_true, y_score)
    return metrics


def run_deployed_model_permutation_control(model_spec, permutation_seeds):
    """Clone the deployed pipeline, retrain on shuffled labels, and score held-out shuffled labels."""
    X, index = load_feature_view(cfg, model_spec["feature_view"])
    if X is None:
        raise FileNotFoundError(f"Feature view not found: {model_spec['feature_view']}")

    # This uses the exact saved train/test split file corresponding to the deployed model.
    _, _, X_train, X_test, y_train, y_test = split_data_for_feature_view(
        X, index, model_spec["split"], model_spec["feature_view"], cfg
    )
    X_train, X_test, y_train, y_test = maybe_subsample(
        X_train,
        X_test,
        y_train,
        y_test,
        max_train_rows=MAX_TRAIN_ROWS,
        max_test_rows=MAX_TEST_ROWS,
    )

    deployed_pipeline = joblib.load(model_spec["model_path"])
    rows = []
    for seed in permutation_seeds:
        y_train_perm, y_test_perm = permute_train_and_test_labels(y_train, y_test, seed)

        # Clone gives the same tuned estimator/preprocessing structure without fitted state.
        permuted_pipeline = clone(deployed_pipeline)
        permuted_pipeline.fit(X_train, y_train_perm)

        y_score = predict_positive_probability(permuted_pipeline, X_test)
        y_pred = (y_score >= 0.5).astype(int)
        metrics = evaluate_predictions(y_test_perm, y_pred, y_score)
        rows.append(
            {
                "model_label": model_spec["label"],
                "feature_view": model_spec["feature_view"],
                "model": model_spec["model"],
                "validation_split": model_spec["split"],
                "permutation_seed": seed,
                "train_n": len(X_train),
                "test_n": len(X_test),
                **metrics,
            }
        )
    return pd.DataFrame(rows)

## Run Permuted-Label Controls

In [ ]:
all_rows = []
for model_spec in DEPLOYED_MODELS:
    print(f"Running permuted-label control: {model_spec['label']} / {model_spec['split']}")
    result = run_deployed_model_permutation_control(model_spec, PERMUTATION_SEEDS)
    all_rows.append(result)

permutation_results = pd.concat(all_rows, ignore_index=True)
out_csv = REPORT_DIR / "deployed_models_permuted_label_control.csv"
permutation_results.to_csv(out_csv, index=False)

print(f"Wrote {out_csv}")
display(permutation_results)

## Summary Table and Figure

In [ ]:
summary = (
    permutation_results.groupby(["model_label", "feature_view", "model", "validation_split"], as_index=False)
    .agg(
        balanced_accuracy_mean=("balanced_accuracy", "mean"),
        balanced_accuracy_std=("balanced_accuracy", "std"),
        roc_auc_mean=("roc_auc", "mean"),
        roc_auc_std=("roc_auc", "std"),
        auprc_mean=("auprc", "mean"),
        auprc_std=("auprc", "std"),
        n_permutations=("permutation_seed", "nunique"),
    )
)
summary_csv = REPORT_DIR / "deployed_models_permuted_label_control_summary.csv"
summary.to_csv(summary_csv, index=False)
print(f"Wrote {summary_csv}")
display(summary)

In [ ]:
# Plot balanced accuracy and ROC-AUC. Values close to 0.5 indicate chance-level behavior.
plot_df = summary.copy()
x = np.arange(len(plot_df))
width = 0.34

fig, ax = plt.subplots(figsize=(10, 5.5), dpi=300)
ax.bar(
    x - width / 2,
    plot_df["balanced_accuracy_mean"],
    width,
    yerr=plot_df["balanced_accuracy_std"],
    capsize=5,
    label="Balanced accuracy",
    color="#2F80C1",
)
ax.bar(
    x + width / 2,
    plot_df["roc_auc_mean"],
    width,
    yerr=plot_df["roc_auc_std"],
    capsize=5,
    label="ROC-AUC",
    color="#8EC5F4",
)
ax.axhline(0.5, color="#333333", linestyle="--", linewidth=1.2, label="Chance")
ax.set_xticks(x)
ax.set_xticklabels(plot_df["model_label"], rotation=20, ha="right")
ax.set_ylabel("Metric value")
ax.set_ylim(0.35, 0.65)
ax.set_title("Permuted-label control for deployed BBB models")
ax.legend(frameon=False)
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()

figure_path = FIGURE_DIR / "deployed_models_permuted_label_control.png"
fig.savefig(figure_path, bbox_inches="tight")
plt.show()
print(f"Wrote {figure_path}")